In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

In [3]:
application = pd.read_csv("Dataset/application_record.csv")
credit = pd.read_csv("Dataset/credit_record.csv")

print("Application Dataset Shape:", application.shape)
print("Credit Dataset Shape:", credit.shape)

application.head()

Application Dataset Shape: (438557, 18)
Credit Dataset Shape: (1048575, 3)


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0


In [4]:
print(credit.head())
print("\nUnique Status Values:")
print(credit['STATUS'].value_counts())

        ID  MONTHS_BALANCE STATUS
0  5001711               0      X
1  5001711              -1      0
2  5001711              -2      0
3  5001711              -3      0
4  5001712               0      C

Unique Status Values:
STATUS
C    442031
0    383120
X    209230
1     11090
5      1693
2       868
3       320
4       223
Name: count, dtype: int64


In [5]:
# Create target variable
credit['target'] = credit['STATUS'].apply(
    lambda x: 1 if x in ['1', '2', '3', '4', '5'] else 0
)

# Merge duplicate IDs by taking the maximum target
credit = credit.groupby('ID')['target'].max().reset_index()

print(credit.head())

        ID  target
0  5001711       0
1  5001712       0
2  5001713       0
3  5001714       0
4  5001715       0


In [6]:
data = application.merge(credit, on='ID', how='inner')

print("Merged Dataset Shape:", data.shape)
data.head()

Merged Dataset Shape: (36457, 19)


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,target
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,1
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,1
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,0


In [7]:
print(data.isnull().sum())

ID                         0
CODE_GENDER                0
FLAG_OWN_CAR               0
FLAG_OWN_REALTY            0
CNT_CHILDREN               0
AMT_INCOME_TOTAL           0
NAME_INCOME_TYPE           0
NAME_EDUCATION_TYPE        0
NAME_FAMILY_STATUS         0
NAME_HOUSING_TYPE          0
DAYS_BIRTH                 0
DAYS_EMPLOYED              0
FLAG_MOBIL                 0
FLAG_WORK_PHONE            0
FLAG_PHONE                 0
FLAG_EMAIL                 0
OCCUPATION_TYPE        11323
CNT_FAM_MEMBERS            0
target                     0
dtype: int64


In [8]:
data = data.dropna()

print("Dataset Shape after removing missing values:", data.shape)

Dataset Shape after removing missing values: (25134, 19)


In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for column in data.columns:
    if data[column].dtype == 'object':
        data[column] = le.fit_transform(data[column])

print(data.head())

        ID  CODE_GENDER  FLAG_OWN_CAR  FLAG_OWN_REALTY  CNT_CHILDREN  \
2  5008806            1             1                1             0   
3  5008808            0             0                1             0   
4  5008809            0             0                1             0   
5  5008810            0             0                1             0   
6  5008811            0             0                1             0   

   AMT_INCOME_TOTAL  NAME_INCOME_TYPE  NAME_EDUCATION_TYPE  \
2          112500.0                 4                    4   
3          270000.0                 0                    4   
4          270000.0                 0                    4   
5          270000.0                 0                    4   
6          270000.0                 0                    4   

   NAME_FAMILY_STATUS  NAME_HOUSING_TYPE  DAYS_BIRTH  DAYS_EMPLOYED  \
2                   1                  1      -21474          -1134   
3                   3                  1      -19110

In [10]:
X = data.drop(['ID', 'target'], axis=1)
y = data['target']

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (25134, 17)
Target Shape: (25134,)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (20107, 17)
Testing Data: (5027, 17)


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, lr_pred))

Logistic Regression Accuracy: 0.8770638551820171

Classification Report:

              precision    recall  f1-score   support

           0       0.88      1.00      0.93      4409
           1       0.00      0.00      0.00       618

    accuracy                           0.88      5027
   macro avg       0.44      0.50      0.47      5027
weighted avg       0.77      0.88      0.82      5027



In [13]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, dt_pred))

Decision Tree Accuracy: 0.87527352297593

Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.95      0.93      4409
           1       0.49      0.32      0.38       618

    accuracy                           0.88      5027
   macro avg       0.70      0.63      0.66      5027
weighted avg       0.86      0.88      0.86      5027



In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.8820370001989258

Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.96      0.93      4409
           1       0.54      0.30      0.39       618

    accuracy                           0.88      5027
   macro avg       0.72      0.63      0.66      5027
weighted avg       0.86      0.88      0.87      5027



In [15]:
import joblib

joblib.dump(rf_model, "credit_card_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [16]:
print(X.columns.tolist())

['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS']


In [17]:
print(X.dtypes)

CODE_GENDER              int64
FLAG_OWN_CAR             int64
FLAG_OWN_REALTY          int64
CNT_CHILDREN             int64
AMT_INCOME_TOTAL       float64
NAME_INCOME_TYPE         int64
NAME_EDUCATION_TYPE      int64
NAME_FAMILY_STATUS       int64
NAME_HOUSING_TYPE        int64
DAYS_BIRTH               int64
DAYS_EMPLOYED            int64
FLAG_MOBIL               int64
FLAG_WORK_PHONE          int64
FLAG_PHONE               int64
FLAG_EMAIL               int64
OCCUPATION_TYPE          int64
CNT_FAM_MEMBERS        float64
dtype: object


In [18]:
print(application.head())

        ID CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  CNT_CHILDREN  \
0  5008804           M            Y               Y             0   
1  5008805           M            Y               Y             0   
2  5008806           M            Y               Y             0   
3  5008808           F            N               Y             0   
4  5008809           F            N               Y             0   

   AMT_INCOME_TOTAL      NAME_INCOME_TYPE            NAME_EDUCATION_TYPE  \
0          427500.0               Working               Higher education   
1          427500.0               Working               Higher education   
2          112500.0               Working  Secondary / secondary special   
3          270000.0  Commercial associate  Secondary / secondary special   
4          270000.0  Commercial associate  Secondary / secondary special   

     NAME_FAMILY_STATUS  NAME_HOUSING_TYPE  DAYS_BIRTH  DAYS_EMPLOYED  \
0        Civil marriage   Rented apartment      -12005 

In [19]:
for col in ['CODE_GENDER',
            'NAME_INCOME_TYPE',
            'NAME_EDUCATION_TYPE',
            'NAME_FAMILY_STATUS',
            'NAME_HOUSING_TYPE',
            'OCCUPATION_TYPE']:
    print("\n", col)
    print(data[col].unique()[:10])


 CODE_GENDER
[1 0]

 NAME_INCOME_TYPE
[4 0 2 3 1]

 NAME_EDUCATION_TYPE
[4 1 2 3 0]

 NAME_FAMILY_STATUS
[1 3 0 2 4]

 NAME_HOUSING_TYPE
[1 4 2 5 0 3]

 OCCUPATION_TYPE
[16 14  0  8 10  4  3  6  1 12]


In [20]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in categorical_columns:
    data[col] = le.fit_transform(data[col])

NameError: name 'categorical_columns' is not defined

In [21]:
from sklearn.preprocessing import LabelEncoder

columns = [
    'CODE_GENDER',
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE'
]

for col in columns:
    le = LabelEncoder()
    le.fit(application[col].fillna("Unknown"))
    print("\n", col)
    for i, value in enumerate(le.classes_):
        print(i, "=", value)


 CODE_GENDER
0 = F
1 = M

 NAME_INCOME_TYPE
0 = Commercial associate
1 = Pensioner
2 = State servant
3 = Student
4 = Working

 NAME_EDUCATION_TYPE
0 = Academic degree
1 = Higher education
2 = Incomplete higher
3 = Lower secondary
4 = Secondary / secondary special

 NAME_FAMILY_STATUS
0 = Civil marriage
1 = Married
2 = Separated
3 = Single / not married
4 = Widow

 NAME_HOUSING_TYPE
0 = Co-op apartment
1 = House / apartment
2 = Municipal apartment
3 = Office apartment
4 = Rented apartment
5 = With parents

 OCCUPATION_TYPE
0 = Accountants
1 = Cleaning staff
2 = Cooking staff
3 = Core staff
4 = Drivers
5 = HR staff
6 = High skill tech staff
7 = IT staff
8 = Laborers
9 = Low-skill Laborers
10 = Managers
11 = Medicine staff
12 = Private service staff
13 = Realty agents
14 = Sales staff
15 = Secretaries
16 = Security staff
17 = Unknown
18 = Waiters/barmen staff


In [22]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

categorical_columns = [
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE'
]

for col in categorical_columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    encoders[col] = le

print("Encoding completed successfully!")

Encoding completed successfully!


In [30]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
import joblib

# Features and target
X = data.drop(["ID", "target"], axis=1)
y = data["target"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Apply SMOTE ONLY on training data
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)

rf_model.fit(X_train, y_train)

# Test Accuracy
pred = rf_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

# Save model
joblib.dump(rf_model, "credit_card_model.pkl")

# Save encoders
joblib.dump(encoders, "encoders.pkl")

print("✅ Model saved successfully!")

Accuracy: 0.820966779391287
✅ Model saved successfully!


In [24]:
print(y.value_counts())
print(y.value_counts(normalize=True))

target
0    22045
1     3089
Name: count, dtype: int64
target
0    0.877099
1    0.122901
Name: proportion, dtype: float64


In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train a new Random Forest model with balanced classes
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

# Make predictions
rf_pred = rf_model.predict(X_test)

# Show results
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

Accuracy: 0.8390690272528347

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.89      0.91      4400
           1       0.39      0.51      0.44       627

    accuracy                           0.84      5027
   macro avg       0.66      0.70      0.67      5027
weighted avg       0.86      0.84      0.85      5027



In [26]:
import joblib

joblib.dump(rf_model, "credit_card_model.pkl")
joblib.dump(encoders, "encoders.pkl")

print("✅ New balanced model saved successfully!")

✅ New balanced model saved successfully!


In [27]:
sample = X.iloc[[0]]

print("Actual Label:", y.iloc[0])

prediction = rf_model.predict(sample)

print("Model Prediction:", prediction[0])

Actual Label: 0
Model Prediction: 0


In [28]:
print(X.columns.tolist())

['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS']
